# Customer Churn Prediction (Classification)
Predict whether a customer will leave the service.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded')

## 1. Load Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/Telco-Customer-Churn.csv'
try:
    df = pd.read_csv(url)
    print('Telco Churn dataset loaded')
except Exception as e:
    print('URL failed, using synthetic data:', e)
    np.random.seed(42)
    n = 2000
    df = pd.DataFrame({
        'gender': np.random.choice(['Male', 'Female'], n),
        'SeniorCitizen': np.random.randint(0, 2, n),
        'tenure': np.random.randint(0, 73, n),
        'PhoneService': np.random.choice(['Yes', 'No'], n),
        'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n),
        'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n),
        'MonthlyCharges': np.round(np.random.uniform(18, 120, n), 2),
        'TotalCharges': np.round(np.random.uniform(18, 8500, n), 2),
        'Churn': np.random.choice(['Yes', 'No'], n, p=[0.27, 0.73])
    })
    df.loc[np.random.choice(n, 50, replace=False), 'TotalCharges'] = np.nan
df.head()

## 2. Data Types of All Columns

In [ ]:
print('Shape:', df.shape)
print()
print(df.dtypes)

## 3. Descriptive Statistics

In [ ]:
df.describe(include='all').T

## 4. Identify and Handle Missing Values

In [ ]:
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
if df['TotalCharges'].dtype == object:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
for col in df.select_dtypes(include='number').columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
print('Missing after handling:', df.isnull().sum().sum())

## 5. Identify and Handle Duplicates

In [ ]:
print(f'Duplicates: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

## 6. Identify and Handle Outliers

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
if 'SeniorCitizen' in num_cols:
    num_cols.remove('SeniorCitizen')
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    before = len(df)
    df = df[(df[col] >= lower) & (df[col] <= upper)]
    removed = before - len(df)
    if removed:
        print(f'  {col}: removed {removed} outliers')
df.reset_index(drop=True, inplace=True)
print(f'Shape after outlier removal: {df.shape}')

## 7. Visualizations & Insights

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

churn_counts = df['Churn'].value_counts()
axes[0].bar(churn_counts.index, churn_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Churn Distribution')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')

df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1], grid=False)
axes[1].set_title('Monthly Charges by Churn')
plt.suptitle('')

for label, grp in df.groupby('Churn')['tenure']:
    axes[2].hist(grp, bins=30, alpha=0.6, label=str(label))
axes[2].set_title('Tenure Distribution by Churn')
axes[2].legend()

plt.tight_layout()
plt.show()

print('Insights:')
print('1. ~27% of customers churned -- class imbalance exists.')
print('2. Churned customers tend to have higher monthly charges.')
print('3. Short-tenure customers are more likely to churn.')

## 8. Encode Categorical Variables & Scale Numerical Features

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
# If Churn is already 0/1 after synthetic generation, map still works

cat_cols = df.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df.drop(columns=['customerID'], errors='ignore', inplace=True)

X = df.drop(columns=['Churn'])
y = df['Churn']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Churn rate in test set: {y_test.mean():.2%}')

## 9. Model Building (Logistic Regression, Decision Tree, Random Forest)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, preds), 3),
        'Precision': round(precision_score(y_test, preds, zero_division=0), 3),
        'Recall':    round(recall_score(y_test, preds, zero_division=0), 3),
        'F1':        round(f1_score(y_test, preds, zero_division=0), 3)
    }
    print(f'{name} trained')
    print(classification_report(y_test, preds, zero_division=0))
    print('-'*40)

## 10. Model Performance Comparison

In [ ]:
results_df = pd.DataFrame(results).T
print(results_df)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
colors = ['#4C72B0', '#DD8452', '#55A868']
for i, metric in enumerate(metrics):
    axes[i].bar(results_df.index, results_df[metric], color=colors)
    axes[i].set_title(f'Model Comparison -- {metric}')
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

best = results_df['F1'].idxmax()
print(f'Best model (by F1): {best} (F1 = {results_df.loc[best, "F1"]})')